MODULE
1. Encoder
2. Token_Representation
3. Entity FFN
4. Span FFN
5. Matching
6. Loss
7. Decder

In [1]:
BATCH_SIZE=32
SPAN_CHANNEL=100

In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer,AutoModel

In [3]:
#Encoder
BackBone_Model="klue/bert-base"

In [4]:
# Input -> (BATCH_SIZE,SEQ_LEN) , Each -> (SEQ_LEN,)
# Output -> (BATCH_SIZE,SEQ_LEN,Hidden_state)
class EncoderModule(nn.Module):
    def __init__(self,backbone_model:str):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(backbone_model)
        self.hidden_size=self.encoder.config.hidden_size
    
    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor)->torch.Tensor:
        outputs=self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_states=outputs.last_hidden_state
        
        return hidden_states

In [5]:
# Token_Representation(h generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,T_MAX,Hidden_size)
class Text_Token_Represntation(nn.Module):
    def __init__(self):
        super().__init__()
        
    def pad_sequence(self,sentences):
        max_len=max(line.size(0) for line in sentences)
        hidden_size=sentences[0].size(1)
        
        pad_sequences=[]
        
        for sent in sentences:
            pad_len=max_len-sent.size(0)
            
            if pad_len>0:
                pad_tensor=torch.zeros(
                    pad_len,
                    hidden_size,
                    device=sent.device,
                    dtype=sent.dtype
                )
                sent=torch.cat([sent,pad_tensor],dim=0)
                
            pad_sequences.append(sent)
        return torch.stack(pad_sequences,dim=0)
        
    def forward(self,hidden_states,text_mask):
        text_vector=[]
        
        for hs,mask in zip(hidden_states,text_mask):
            h_vector=hs[mask.bool()]
            text_vector.append(h_vector)
        
        padded_text_vectors=self.pad_sequence(text_vector)    
    
        return padded_text_vectors

In [6]:
# Test
hidden_states=torch.randn(2,6,4)
text_mask=torch.Tensor([
    [0,0,1,1,1,0],
    [0,1,1,0,0,0]
])
model=Text_Token_Represntation()
output=model(hidden_states,text_mask)


print("hidden_state shape : ",hidden_states.shape)
print("text_mask shape : ",text_mask.shape)
print("output shape : ",output.shape)
print(hidden_states)
#print("\n")
print(output)


hidden_state shape :  torch.Size([2, 6, 4])
text_mask shape :  torch.Size([2, 6])
output shape :  torch.Size([2, 3, 4])
tensor([[[ 0.6564,  0.8004,  0.8703,  0.2109],
         [-1.7035,  0.2229, -0.0835, -0.7445],
         [-0.0637,  0.8034,  0.6704,  0.9581],
         [-0.0707,  1.4265,  0.8721, -0.3707],
         [-0.7276, -0.4718,  0.8032, -0.7055],
         [ 0.8584,  0.1654, -1.2097,  0.4972]],

        [[-0.0080, -0.6817, -0.0847,  1.6239],
         [-0.0098, -0.2443, -0.4819,  0.8603],
         [ 0.8658, -0.3851,  1.1351,  1.7761],
         [-1.8011, -0.5517,  0.6790, -2.6698],
         [ 0.8281, -2.0367,  1.1110, -0.1272],
         [ 0.8070, -0.8428,  0.2901, -1.1552]]])
tensor([[[-0.0637,  0.8034,  0.6704,  0.9581],
         [-0.0707,  1.4265,  0.8721, -0.3707],
         [-0.7276, -0.4718,  0.8032, -0.7055]],

        [[-0.0098, -0.2443, -0.4819,  0.8603],
         [ 0.8658, -0.3851,  1.1351,  1.7761],
         [ 0.0000,  0.0000,  0.0000,  0.0000]]])


In [40]:
# Token_Representation(p generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,M,Hidden_size) (M=[ENT] count)
# batch 단위로 돌리려면 p_vector 길이 같아야함
class ENT_Token_Representation(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,hidden_states,ent_mask):
        ent_vectors=[]
        
        for hs,mask in zip(hidden_states,ent_mask):
            p_vector=hs[mask.bool()]
            ent_vectors.append(p_vector)
        max_ent=max(v.size(0) for v in ent_vectors)
        hidden_size=hidden_states.size(-1)
        
        padded_ent_vectors=[]
        padded_ent_masks=[]
        
        for v in ent_vectors:
            m=v.size(0)
            pad_len=max_ent-m
            
            if pad_len>0:
                pad_tensor=torch.zeros(
                    pad_len,
                    hidden_size,
                    device=v.device,
                    dtype=v.dtype
                )
                v=torch.cat([v,pad_tensor],dim=0)
            valid_mask=torch.cat([
                torch.ones(m,device=v.device,dtype=torch.bool),
                torch.zeros(pad_len,device=v.device,dtype=torch.bool)
            ],dim=0)
            padded_ent_vectors.append(v)
            padded_ent_masks.append(valid_mask)
            
        ent_vectors=torch.stack(padded_ent_vectors,dim=0)
        ent_valid_mask=torch.stack(padded_ent_masks,dim=0)
        
        return ent_vectors,ent_valid_mask

In [8]:
# Test
hidden_states=torch.randn(2,8,4)
ent_token_mask=torch.tensor([
    [0,1,0,1,0,1,0,0],
    [0,1,0,1,0,1,0,0]
])
model=ENT_Token_Representation()
output=model(hidden_states,ent_token_mask)

print("hidden_states shape : ",hidden_states.shape)
print(hidden_states)
print("ent_token_mask shape : ",ent_token_mask.shape)
print("output shape : ",output.shape)
print(output)

hidden_states shape :  torch.Size([2, 8, 4])
tensor([[[ 0.5720,  0.7448,  0.3588,  0.4641],
         [-1.3084,  0.7078, -0.8579,  1.4629],
         [-0.8852, -0.0661, -1.1135,  1.0467],
         [-0.0718,  0.1253, -0.0759,  1.5085],
         [ 0.1437,  0.1672, -0.2342, -0.1505],
         [-0.0404, -1.3772,  0.6294,  0.6546],
         [-2.1871,  1.2068,  0.5155,  1.5720],
         [ 0.1643, -0.7520, -0.2540, -0.8552]],

        [[-0.4615,  0.3403, -1.0646,  1.5771],
         [ 0.7107, -0.3636,  0.1138,  1.2593],
         [ 1.4591,  0.7334, -0.3299, -0.2500],
         [-0.6449,  0.0108, -0.2176,  0.2375],
         [-1.2460,  0.3582,  0.0249, -1.1924],
         [ 0.4333, -0.2403,  0.3036, -0.3730],
         [ 0.8304,  0.3873,  1.3574, -1.5598],
         [ 0.0833,  0.5223, -1.3227,  0.1058]]])
ent_token_mask shape :  torch.Size([2, 8])
output shape :  torch.Size([2, 3, 4])
tensor([[[-1.3084,  0.7078, -0.8579,  1.4629],
         [-0.0718,  0.1253, -0.0759,  1.5085],
         [-0.0404, -1.37

In [9]:
class Linear(nn.Module):
    def __init__(self,input_x,output_y,bias=True):
        super().__init__()
        
        self.w=nn.Parameter(torch.randn(output_y,input_x))
        
        if bias:
            self.bias=nn.Parameter(torch.randn(output_y))
        else:
            self.bias=None
    def forward(self,x):
        out=x @ self.w.T
        
        if self.bias is not None:
            out+=self.bias
        return out

In [10]:
# test
linear=Linear(768,256)
x=torch.randn(32,10,768)
y=linear(x)
print(y.shape)

torch.Size([32, 10, 256])


In [11]:
#class ReLU(nn.Module):
#    def __init__(self):
#        super().__init__()
#    def forward(self,x):
#        if(x>=0):
#            return x
#        else:
#            return 0
# 이거는 Tensor 각각을 보지않아서 동작안함
class ReLU(nn.Module):
    def forward(self,x):
        return x*(x>0)

In [12]:
# test
activation=ReLU()
print(activation(-2))

0


In [13]:
class Sequential(nn.Module):
    def __init__(self,*layers):
        super().__init__()
        self.layers=nn.ModuleList(layers)
        
    def forward(self,x):
        for layer in self.layers:
            x=layer(x)

        return x

In [14]:
# test
model=Sequential(
    Linear(4,8),
    ReLU(),
    #Linear(8,2)
)
x=torch.randn(3,4)
y=model(x)
print(y.shape)
print(x)
print(y)

torch.Size([3, 8])
tensor([[ 1.2430, -1.8890, -0.9164,  1.3659],
        [-1.1823, -0.2634, -0.0499,  0.4010],
        [ 0.6149, -0.3861,  0.8979, -0.2329]])
tensor([[-0.0000, -0.0000, 1.1074, -0.0000, -0.0000, -0.0000, 0.6948, 7.1661],
        [-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 1.3013, -0.0000],
        [0.0566, -0.0000, 0.5218, 0.1388, 0.2568, -0.0000, -0.0000, 0.2425]],
       grad_fn=<MulBackward0>)


In [15]:
# Entity FFN
# input -> (Batch_size,M,d_model)
# output -> (Batch_size,M,d_model)
class Entity_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        
        self.net=Sequential(
            Linear(d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )
    def forward(self,x):
        output=self.net(x)
        return output

In [16]:
# test
model=Entity_Representation(768,1024)
x=torch.randn(32,10,768)
y=model(x)
print(x.shape)
print(y.shape)

torch.Size([32, 10, 768])
torch.Size([32, 10, 768])


In [17]:
# Span FFN
class Span_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.net=Sequential(
            Linear(2*d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )
    
    def forward(self,start_token,end_token):
        span=torch.cat([start_token,end_token],dim=-1)
        output=self.net(span)
        return output

In [18]:
# test
model=Span_Representation(768,1024)
start_token=torch.randn(32,20,768)
end_token=torch.randn(32,20,768)
y=model(start_token,end_token)
print("start shape : ",start_token.shape)
print("end shape : ",end_token.shape)
print(y.shape)

start shape :  torch.Size([32, 20, 768])
end shape :  torch.Size([32, 20, 768])
torch.Size([32, 20, 768])


In [19]:
class Sigmoid(nn.Module):
    def forward(self,x):
        output=1/(1+torch.exp(-x))
        return output

In [ ]:
# sigmoid를 만들어 BCELoss를 쓰면 연산 불안정해짐, 그래서 BCEWithLogitLoss 써야함
class Matching(nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid=Sigmoid()
    def forward(self,entity,span):
        entity=entity.transpose(-1,-2)
        output=span@entity
        #score=self.sigmoid(output)
        return output

In [21]:
# test
B,E,N,d=2,5,7,4
entity=torch.randn(B,E,d)
span=torch.randn(B,N,d)
model=Matching()
score=model(entity,span)
print(score.shape)
print(score)

torch.Size([2, 7, 5])
tensor([[[0.9137, 0.1233, 0.9356, 0.2001, 0.7636],
         [0.3566, 0.8006, 0.2743, 0.3393, 0.6296],
         [0.6037, 0.5746, 0.6514, 0.1989, 0.7209],
         [0.9382, 0.9590, 0.0691, 0.7422, 0.7913],
         [0.3080, 0.2654, 0.8516, 0.3682, 0.3349],
         [0.9966, 0.3807, 0.4041, 0.9281, 0.7282],
         [0.1268, 0.7402, 0.5748, 0.1172, 0.5912]],

        [[0.7679, 0.1965, 0.7844, 0.5639, 0.1350],
         [0.6010, 0.3327, 0.5634, 0.5584, 0.5998],
         [0.1769, 0.9193, 0.2996, 0.4930, 0.3487],
         [0.5069, 0.6190, 0.8876, 0.8410, 0.1420],
         [0.4762, 0.4799, 0.4856, 0.4928, 0.7863],
         [0.5974, 0.3259, 0.2474, 0.4417, 0.4293],
         [0.7171, 0.2148, 0.7931, 0.5339, 0.5042]]])


In [22]:
class BCEloss(nn.Module):
    def forward(self,pred,ans):
        eps=1e-12
        pred=torch.clamp(pred,eps,1-eps)
        loss=-(ans*torch.log(pred)+(1-ans)*torch.log(1-pred))
        output=loss.mean()
        return output

In [23]:
# test
target=torch.randint(0,2,(B,N,E)).float()
loss_f=BCEloss()
loss=loss_f(score,target)
print(loss)

tensor(0.7495)


In [36]:
class Decoder(nn.Module):
    def __init__(self,entity_types,threshold=0.5,mode="flat"):
        super().__init__()
        self.entity_types=entity_types
        self.threshold=threshold
        self.mode=mode
    
    def overlapping(self,span1,span2):
        s1,e1=span1
        s2,e2=span2
        
        return not(e1<s2 or e2<s1)
    
    def partial_overlap(self,span1,span2):
        s1,e1=span1
        s2,e2=span2
        
        overlap=not(e1<s2 or e2<s1)
        if not overlap:
            return False
        
        nested=(s1<=s2 and e2<=e1) or (s2<=s1 and e1 <=e2)
        
        return not nested
    
    def decode_one(self,scores,span_indices,span_mask):
        candiates=[]
        
        N_span,E=scores.shape
        
        for n in range(N_span):
            if not span_mask[n]:
                continue
            for e in range(E):
                score=scores[n, e].item()
                
                if score>self.threshold:
                    start,end=span_indices[n]
                    label=self.entity_types[e]
                    candiates.append((start,end,label,score))
        
        candiates.sort(key=lambda x:x[3],reverse=True)
        
        selected=[]
        
        for cand in candiates:
            c_start,c_end,c_label,c_score=cand
            conflict=False
            
            for sel in selected:
                s_start,s_end,_,_=sel
                
                if self.mode=="flat":
                    if self.overlapping((c_start,c_end),(s_start,s_end)):
                        conflict=True
                        
                        break
                    
                elif self.mode=="nested":
                    if self.partial_overlap((c_start,c_end),(s_start,s_end)):
                        conflict=True
                        
                        break
                    
            if not conflict:
                selected.append(cand)
                
        return selected
    
    def decode_batch(self,scores,span_indices,span_mask):
        results=[]
        B=scores.shape[0]
        
        for b in range(B):
            results.append(self.decode_one(scores[b],span_indices,span_mask[b]))
            
        return results

In [37]:
# test
decoder=Decoder(
    entity_types=["PER","ORG"],
    threshold=0.5,
    mode="flat"
)

scores=torch.tensor([
    [0.9,0.1],
    [0.8,0.2],
    [0.3,0.7],
    [0.6,0.4],
],dtype=torch.float32)

span_indices=[
    (0,0),
    (0,1),
    (2,2),
    (1,2),
]
print(scores.shape)
result=decoder.decode_one(scores,span_indices)
print(result)

torch.Size([4, 2])


TypeError: Decoder.decode_one() missing 1 required positional argument: 'span_mask'

In [26]:
# test
assert len(result)==2
assert result[0][0:3]==(0,0,"PER")
assert result[1][0:3]==(2,2,"ORG")
print("pass")

pass


In [27]:
# test
scores_batch = torch.tensor([
    [
        [0.9, 0.1],
        [0.8, 0.2],
        [0.3, 0.7],
        [0.6, 0.4],
    ],
    [
        [0.2, 0.9],
        [0.1, 0.4],
        [0.7, 0.2],
        [0.3, 0.8],
    ]
], dtype=torch.float32)

decoder = Decoder(
    entity_types=["PER", "ORG"],
    threshold=0.5,
    mode="flat"
)

results = decoder.decode_batch(scores_batch, span_indices)
print(results)

[[(0, 0, 'PER', 0.8999999761581421), (2, 2, 'ORG', 0.699999988079071)], [(0, 0, 'ORG', 0.8999999761581421), (1, 2, 'ORG', 0.800000011920929)]]


In [28]:
decoder_nested = Decoder(
    entity_types=["PER", "ORG"],
    threshold=0.5,
    mode="nested"
)

scores = torch.tensor([
    [0.95, 0.1],   # (0,2) -> PER
    [0.85, 0.2],   # (1,1) -> PER   : (0,2) 안에 완전 포함
    [0.80, 0.1],   # (1,3) -> PER   : (0,2)와 partial overlap
], dtype=torch.float32)

span_indices = [
    (0, 2),
    (1, 1),
    (1, 3),
]

result_nested = decoder_nested.decode_one(scores, span_indices)
print(result_nested)

[(0, 2, 'PER', 0.949999988079071), (1, 1, 'PER', 0.8500000238418579)]


In [ ]:
class GLiNER(nn.Module):
    def __init__(self,backbone_model,entity_types,d_ff,max_span_width=10,threshold=0.5,mode="flat"):
        super().__init__()
        
        self.entity_types=entity_types
        self.num_entity_types=len(entity_types)
        self.max_span_width=max_span_width
        
        self.encoder=EncoderModule(backbone_model)
        d_model=self.encoder.hidden_size
        
        self.text_token_rep=Text_Token_Represntation()
        self.ent_token_rep=ENT_Token_Representation()
        
        self.ent_rep=Entity_Representation(d_model,d_ff)
        self.span_rep=Span_Representation(d_model,d_ff)
        
        self.sigmoid=Sigmoid()
        self.matching=Matching()
        
        self.decoder=Decoder(
            entity_types=entity_types,
            threshold=threshold,
            mode=mode
        )
        
    def build_span_representation(self,text_embeddings,text_mask):
        B,T,H=text_embeddings.shape
        
        span_embeddings=[]
        span_indices=[]
        span_masks=[]
        
        for start in range(T):
            max_end=min(T,start+self.max_span_width)
            for end in range(start,max_end):
                start_token=text_embeddings[:,start,:]
                end_token=text_embeddings[:,end,:]
                
                span_emb=self.span_rep(start_token,end_token)
                span_embeddings.append(span_emb)
                span_indices.append((start,end))
                
                # add
                valid_span=text_mask[:,start] & text_mask[:,end]
                span_masks.append(valid_span)
                
        span_embeddings=torch.stack(span_embeddings,dim=1)
        span_mask=torch.stack(span_masks,dim=1)
        
        return span_embeddings,span_indices,span_mask
        
    def forward(self,input_ids,attention_mask,text_mask,ent_mask):
        hidden_states=self.encoder(input_ids,attention_mask)
        
        text_embeddings=self.text_token_rep(hidden_states,text_mask)
        ent_embeddings,ent_valid_mask=self.ent_token_rep(hidden_states,ent_mask)
        
        ent_vectors,ent_valid_mask=self.ent_rep(ent_embeddings)
        span_vectors,span_indices,span_mask=self.build_span_representation(text_embeddings,text_mask)
        logits=self.matching(ent_vectors,span_vectors)
        
        ent_valid_mask=ent_valid_mask.unsqueeze(1)
        logits=logits.maske_fill(~ent_valid_mask,-1e9)
        return logits,span_indices,span_mask
    
    @torch.no_grad()
    def predict(self,input_ids,attention_mask,text_mask,ent_mask):
        logits,span_indices,span_mask=self.forward(input_ids=input_ids,attention_mask=attention_mask,text_mask=text_mask,ent_mask=ent_mask)
        
        scores=self.Sigmoid(logits)
        results=self.decoder.decode_batch(scores,span_indices,span_mask)
        
        return results

In [30]:
model=GLiNER(
    backbone_model=BackBone_Model,
    entity_types=["person","location","organization"],
    d_ff=512,
    max_span_width=10,
    threshold=0.5,
    mode="flat"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
